# NetGuard: Intelligent Network Anomaly Detection
## Hybrid Machine Learning Model: Isolation Forest + Autoencoder
### Project: Network Anomaly Detection using Isolation Forest with Autoencoder Hybrid Model
**Course**: Machine Learning for Electrical / Computer Engineers  
**Objective**: Detect malicious and abnormal network traffic by combining Deep Learning (Autoencoder reconstruction) with Anomaly Detection (Isolation Forest).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

## 1. Data Ingestion & Preprocessing

In [ ]:
dataset_path = '../datasets/nsl_kdd_sample.csv'
df = pd.read_csv(dataset_path)
print("Dataset Shape:", df.shape)
df.head()

## 2. Autoencoder Model Architecture (Deep Learning)
Learns normal network behavior by encoding inputs into a compressed latent space and reconstructing them. Unseen attacks yield high Mean Squared Error (MSE).

In [ ]:
# Preprocess features
scaler = StandardScaler()
X = df[['duration', 'src_bytes', 'dst_bytes', 'packet_count']].values
X_scaled = scaler.fit_transform(X)

# Autoencoder: Input(4) -> Hidden(16) -> Latent(3) -> Hidden(16) -> Output(4)
autoencoder = MLPRegressor(hidden_layer_sizes=(16, 3, 16), activation='relu', solver='adam', max_iter=200, random_state=42)
autoencoder.fit(X_scaled, X_scaled)
reconstructions = autoencoder.predict(X_scaled)
mse_errors = np.mean(np.square(X_scaled - reconstructions), axis=1)
print("Autoencoder Mean MSE:", np.mean(mse_errors))

## 3. Isolation Forest Model (Tree-based Anomaly Detection)
Isolates anomalous instances by randomly partitioning feature space.

In [ ]:
iso_forest = IsolationForest(n_estimators=150, contamination=0.1, random_state=42)
iso_forest.fit(X_scaled)
raw_scores = iso_forest.decision_function(X_scaled)
norm_iso = (np.max(raw_scores) - raw_scores) / (np.max(raw_scores) - np.min(raw_scores))
print("Isolation Forest Normalized Scores Range:", np.min(norm_iso), "to", np.max(norm_iso))

## 4. Hybrid Scoring Formula
$$ \text{Hybrid Score} = 0.55 \times \text{Norm(Isolation Forest)} + 0.45 \times \text{Norm(Autoencoder MSE)} $$
Alert Threshold = 0.50

In [ ]:
norm_ae = (mse_errors - np.min(mse_errors)) / (np.max(mse_errors) - np.min(mse_errors) + 1e-6)
hybrid_scores = 0.55 * norm_iso + 0.45 * norm_ae
predictions = np.where(hybrid_scores >= 0.5, 1, 0)
print("Total Samples:", len(predictions))
print("Anomalies Detected:", np.sum(predictions))
print("Normal Traffic:", len(predictions) - np.sum(predictions))